# 第一篇：数据结构（就两个核心）
概念|Python类比|解释
---|---|---
Series|	带标签的列表|	表格中的一列数据
DataFrame	|字典组成的列表|	一张完整表格

In [2]:
import pandas as pd
df = pd.DataFrame({
    '姓名': ['张三', '李四', '王五', '赵六'],
    '年龄': [25, 30, 35, 28],
    '薪资': [8000, 15000, 20000, 12000],
    '部门': ['销售', '研发', '研发', '市场']
})
print(df)
# 从csv文件读：
# df = pd.read_csv('data.csv')

   姓名  年龄     薪资  部门
0  张三  25   8000  销售
1  李四  30  15000  研发
2  王五  35  20000  研发
3  赵六  28  12000  市场


# 第二篇：只看一眼数据（必背三板斧）
拿到数据后不要直接写逻辑，先做这三步诊断：

In [3]:
# 1. 看头和尾，确认读对了没
print(df.head(2))      # 前两行
print(df.tail(1))      # 后一行

# 2. 看结构信息（极其重要！用于看空值和类型）
print()
df.info()       
# 输出会告诉你：哪列有缺失值？薪资是不是数字（还是字符串）？

# 3. 看统计摘要（找出异常值）
print()
df.describe()   
# 关注 min/max，如果年龄 min 是 -1，说明数据脏了。

   姓名  年龄     薪资  部门
0  张三  25   8000  销售
1  李四  30  15000  研发
   姓名  年龄     薪资  部门
3  赵六  28  12000  市场

<class 'pandas.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   姓名      4 non-null      str  
 1   年龄      4 non-null      int64
 2   薪资      4 non-null      int64
 3   部门      4 non-null      str  
dtypes: int64(2), str(2)
memory usage: 260.0 bytes



,年龄,薪资
count,4.000000,4.000000
mean,29.500000,13750.000000
std,4.203173,5057.996968
min,25.000000,8000.000000
25%,27.250000,11000.000000
50%,29.000000,13500.000000
75%,31.250000,16250.000000
max,35.000000,20000.000000


# 第三篇：选取数据（最难但也最常用）
机器学习中处理特征（X）和标签（y）全靠这几种选取方式。

In [4]:
# 1. 选一列（返回 Series，相当于一维数组）
print(df['年龄'])       



# 一句话写法（最常用）
df[df['薪资'] > 10000]
df[(df['薪资'] > 10000) & (df['年龄'] < 35)]  # 且: &; 或: |

0    25
1    30
2    35
3    28
Name: 年龄, dtype: int64


,姓名,年龄,薪资,部门
1,李四,30,15000,研发
3,赵六,28,12000,市场


In [5]:
# 2. 选多列（返回 DataFrame，机器学习的 X 通常这样做）
# 注意里面有两个中括号！

print(df[['年龄', '薪资']])

   年龄     薪资
0  25   8000
1  30  15000
2  35  20000
3  28  12000


In [6]:
# 3. 按条件筛选行（SQL 里的 WHERE）
# 找出所有研发部的人
mask = df['部门'] == '研发'  # 生成一列 True/False
df[mask]                    # 塞回 df 取行

,姓名,年龄,薪资,部门
1,李四,30,15000,研发
2,王五,35,20000,研发


In [7]:
# 一句话写法（最常用）
df[df['薪资'] > 10000]
df[(df['薪资'] > 10000) & (df['年龄'] < 35)]  # 且: &; 或: |

,姓名,年龄,薪资,部门
1,李四,30,15000,研发
3,赵六,28,12000,市场


# 第四篇：缺失值处理（模型报错的万恶之源）

In [8]:
# 查看哪儿有空格（NaN）
df.isnull().sum()

# 方案 A：直接删掉缺失数据的行（简单粗暴）
df_clean = df.dropna()

# 方案 B：填个数（机器学习最推荐）
# 用平均值填充数值列
avg_salary = df['薪资'].mean()
df['薪资'] = df['薪资'].fillna(avg_salary)

# 用“无”填充文字列
df['部门'] = df['部门'].fillna('未知')

# 第五篇：特征工程常用操作（Groupby 与 Apply）

1. 分组聚合（Groupby）
类比：Excel 的数据透视表 / SQL 的 GROUP BY

In [9]:
# 问题：每个部门的平均薪资是多少？
df.groupby('部门')['薪资'].mean()
# 输出一行结果，索引是部门名

部门
市场    12000.0
研发    17500.0
销售     8000.0
Name: 薪资, dtype: float64

2. 新增列（特征构造）
机器学习里经常需要算个 BMI 或 总价什么的

In [10]:
df['税后估算'] = df['薪资'] * 0.8
print(df.head())

   姓名  年龄     薪资  部门     税后估算
0  张三  25   8000  销售   6400.0
1  李四  30  15000  研发  12000.0
2  王五  35  20000  研发  16000.0
3  赵六  28  12000  市场   9600.0


3. 映射 / 替换（Map / Replace）
   

In [11]:
# 把研发 -> 1, 销售 -> 2, 市场 -> 3
mapping = {'研发': 1, '销售': 2, '市场': 3}
df['部门编码'] = df['部门'].map(mapping)
print(df)

   姓名  年龄     薪资  部门     税后估算  部门编码
0  张三  25   8000  销售   6400.0     2
1  李四  30  15000  研发  12000.0     1
2  王五  35  20000  研发  16000.0     1
3  赵六  28  12000  市场   9600.0     3


# 第六篇：Pandas 与 机器学习的连接处（关键！）
模型（如 sklearn）只吃 Numpy 数组 或者 纯数字表格。

In [12]:
# 假设 df 是处理干净的数据框

# 1. 提取特征 X (DataFrame)
# 排除掉目标列，只保留输入特征
features = ['年龄', '薪资', '部门编码'] 
X = df[features]         # 此时 X 是 DataFrame

# 2. 提取标签 y (Series)
y = df['是否离职']       # 这是你要预测的列

# 3. 转为 numpy 给 sklearn 用
X_array = X.values       # 或者 X.to_numpy()
y_array = y.values

KeyError: '是否离职'